# TSIC I. Ingeniería de Datos - Limpieza de Datos con Spark
> 2025/10/09


### Importante: Elegir como kernel el 'PySpark Env'

In [6]:
# Installing Spark (ya no es necesario pues se crea el ambiente)
# %pip install -q pyspark

In [7]:
# Display the uploaded file
!ls

00-python-env.sh  01-limpieza-spark.ipynb  venv_spark


In [8]:
# Creating the Spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DatosExamen").getOrCreate()

In [9]:
# Save the file contents to a data frame
df = spark.read.csv('../data-sets/DatosExamen.csv', header=True, inferSchema=True, quote='"', escape='"')

In [10]:
# Unfold the data frame
df.show()

# Dataframe format
df.head()

+--------------------+--------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|              Nombre|    Edad| Lugar de Nacimiento|            Promedio| Fecha de Nacimiento|          Donde Vivo|   Trabajo|       Dónde Trabajo|            Semestre|          Ingeniería|                Dato|    Artista Favorito|           3 hobbies|    Con cuantos vivo|3 Materias Favoritas|
+--------------------+--------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|   Aldo Abad Vásquez|      25|Camerino Z. Mendo...|                8,78|23 de octubre de ...|Copilco El Alto, ...|

Row(Nombre='Aldo Abad Vásquez', Edad='25', Lugar de Nacimiento='Camerino Z. Mendoza,Veracruz', Promedio='8,78', Fecha de Nacimiento='23 de octubre de 1999', Donde Vivo='Copilco El Alto, Coyoacán, CDMX', Trabajo='No', Dónde Trabajo=None, Semestre='Ya debería haber terminado', Ingeniería='La aplicación de conocimientos y técnicas a un fin.', Dato='La unidad mínima de información.', Artista Favorito='Ado', 3 hobbies='Tocar guitarra, cocinar y modding de Minecraft', Con cuantos vivo='7', 3 Materias Favoritas='Bases de Datos Distribuidas, Minería de Datos y Taller Socio-humanístico: Liderazgo')

## Bibliotecas

In [11]:
# Import necessary functions for DataFrame manipulation and transformations
from pyspark.sql.functions import regexp_replace, regexp_extract, col, when, lower
from pyspark.sql.functions import split, trim, substring, concat, lit
from pyspark.sql.functions import coalesce
from pyspark.sql.functions import udf, initcap
from datetime import datetime
from pyspark.sql.types import StringType, ArrayType

## Nombre

In [12]:
# Rename the 'Nombre' column to 'name'
name_df = df.withColumnRenamed("Nombre", "name")

name_df.show()

+--------------------+--------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|                name|    Edad| Lugar de Nacimiento|            Promedio| Fecha de Nacimiento|          Donde Vivo|   Trabajo|       Dónde Trabajo|            Semestre|          Ingeniería|                Dato|    Artista Favorito|           3 hobbies|    Con cuantos vivo|3 Materias Favoritas|
+--------------------+--------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|   Aldo Abad Vásquez|      25|Camerino Z. Mendo...|                8,78|23 de octubre de ...|Copilco El Alto, ...|

## Edad

In [13]:
# Remove non-numeric characters from 'Edad' column and cast to integer
age_df_cleaned = name_df.withColumn("Edad_cleaned", regexp_replace(col("Edad"), "[^0-9]", ""))
age_df_casted = age_df_cleaned.withColumn("age", col("Edad_cleaned").cast("int"))

# Drop the original 'Edad' column and the temporary cleaned column
age_df = age_df_casted.drop("Edad", "Edad_cleaned")

# Show the updated DataFrame
age_df.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+
|                name| Lugar de Nacimiento|            Promedio| Fecha de Nacimiento|          Donde Vivo|   Trabajo|       Dónde Trabajo|            Semestre|          Ingeniería|                Dato|    Artista Favorito|           3 hobbies|    Con cuantos vivo|3 Materias Favoritas|age|
+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+
|   Aldo Abad Vásquez|Camerino Z. Mendo...|                8,78|23 de octubre de ...|Copilco El Alto, ...|        No|             

## Lugar de Nacimiento

In [14]:
def normalize_state(s):
    mapping = {
        'CDMX': 'Ciudad de México',
        'D.F': 'Ciudad de México',
        'Ciudad de México': 'Ciudad de México',
        'Edo. Méx': 'Estado de México',
        'Estado de México': 'Estado de México',
        'Veracruz': 'Veracruz',
        'Oaxaca': 'Oaxaca',
        'Hidalgo': 'Hidalgo',
    }
    for k, v in mapping.items():
        if k in s:
            return v
    return None

# Register the UDF
state_normalize_udf = udf(normalize_state, StringType())

# Apply the UDF to the 'Lugar de Nacimiento' column and rename it
state_df_normalized = age_df.withColumn("state", state_normalize_udf(col("Lugar de Nacimiento")))

# Drop the original 'Lugar de Nacimiento' column
state_df = state_df_normalized.drop("Lugar de Nacimiento")

# Show the updated DataFrame
state_df.show()

[Stage 11:>                                                         (0 + 1) / 1]

+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+----------------+
|                name|            Promedio| Fecha de Nacimiento|          Donde Vivo|   Trabajo|       Dónde Trabajo|            Semestre|          Ingeniería|                Dato|    Artista Favorito|           3 hobbies|    Con cuantos vivo|3 Materias Favoritas|age|           state|
+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+----------------+
|   Aldo Abad Vásquez|                8,78|23 de octubre de ...|Copilco El Alto, ...|        No|                NULL|Ya debería haber ...|La a

## Promedio

In [15]:
# Remove non-numeric characters except comma, replace comma with period, and cast to double
grade_df_cleaned = state_df.withColumn("Promedio_cleaned", regexp_replace(col("Promedio"), "[^0-9,.]", ""))
grade_df_replaced = grade_df_cleaned.withColumn("Promedio_replaced", regexp_replace(col("Promedio_cleaned"), ",", "."))
grade_df_casted = grade_df_replaced.withColumn("grade_average", col("Promedio_replaced").cast("double"))

# Drop the original 'Promedio' column and temporary cleaned columns
grade_df = grade_df_casted.drop("Promedio", "Promedio_cleaned", "Promedio_replaced")

# Show the updated DataFrame
grade_df.show()

+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+----------------+-------------+
|                name| Fecha de Nacimiento|          Donde Vivo|   Trabajo|       Dónde Trabajo|            Semestre|          Ingeniería|                Dato|    Artista Favorito|           3 hobbies|    Con cuantos vivo|3 Materias Favoritas|age|           state|grade_average|
+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+----------------+-------------+
|   Aldo Abad Vásquez|23 de octubre de ...|Copilco El Alto, ...|        No|                NULL|Ya debería haber ...|La aplicación de ...|La unidad mínima ...|    

## Fecha de Nacimiento

In [16]:
# Define a function to parse and format the date
def parse_and_format_date(date_str):
    if date_str is None:
        return None

    # Recursive function to replace Spanish month names
    def replace_months(s, month_items, index=0):
        if index >= len(month_items):
            return s
        es, en = month_items[index]
        s_replaced = s.replace(es, en)
        return replace_months(s_replaced, month_items, index + 1)

    date_str_cleaned = date_str.lower().replace(' de ', ' ').replace(' del ', ' ')
    month_map = {
        'enero': 'January', 'febrero': 'February', 'marzo': 'March', 'abril': 'April',
        'mayo': 'May', 'junio': 'June', 'julio': 'July', 'agosto': 'August',
        'septiembre': 'September', 'octubre': 'October', 'noviembre': 'November', 'diciembre': 'December'
    }
    date_str_cleaned = replace_months(date_str_cleaned, list(month_map.items()))

    # Function to try parsing with different formats
    def try_formats(date_string, formats, index=0):
        if index >= len(formats):
            return None
        try:
            return datetime.strptime(date_string, formats[index]).strftime('%d/%m/%Y')
        except ValueError:
            return try_formats(date_string, formats, index + 1)

    date_formats = ['%d/%m/%Y', '%d %B %Y', '%d/%b/%Y', '%d/%B/%Y']
    return try_formats(date_str_cleaned, date_formats)


# Register the UDF
parse_and_format_udf = udf(parse_and_format_date, StringType())

# Apply the UDF to the 'Fecha de Nacimiento' column and rename it
birthday_df_normalized = grade_df.withColumn("date_of_birth", parse_and_format_udf(col("Fecha de Nacimiento")))

# Drop the original 'Fecha de Nacimiento' column
birthday_df = birthday_df_normalized.drop("Fecha de Nacimiento")

# Show the updated DataFrame
birthday_df.show()

[Stage 13:>                                                         (0 + 1) / 1]

+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+----------------+-------------+-------------+
|                name|          Donde Vivo|   Trabajo|       Dónde Trabajo|            Semestre|          Ingeniería|                Dato|    Artista Favorito|           3 hobbies|    Con cuantos vivo|3 Materias Favoritas|age|           state|grade_average|date_of_birth|
+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+----------------+-------------+-------------+
|   Aldo Abad Vásquez|Copilco El Alto, ...|        No|                NULL|Ya debería haber ...|La aplicación de ...|La unidad mínima ...|                 Ado|Tocar guitarra, c...|    

## Dónde Vivo

In [ ]:
# Clean up the 'Donde Vivo' column
current_residence_df_cleanned = birthday_df.withColumn(
    "Donde Vivo",
    regexp_replace(lower(col("Donde Vivo")), r"^(en )", "")
)

# Mapping of known districts to their status
district_to_state = {
    "coyoacán": "Ciudad de México",
    "benito juárez": "Ciudad de México",
    "venustiano carranza": "Ciudad de México",
    "chimalhuacán": "Estado de México",
    "ocoyoacac": "Estado de México",
    "tizayuca": "Hidalgo"
}

# Extract the main district based on known matches
current_residence_df_disctricts = current_residence_df_cleanned.withColumn(
    "current_district",
    when(col("Donde Vivo").contains("coyoacan"), lit("Coyoacán"))
    .when(col("Donde Vivo").contains("coyoacán"), lit("Coyoacán"))
    .when(col("Donde Vivo").contains("benito juárez"), lit("Benito Juárez"))
    .when(col("Donde Vivo").contains("venustiano carranza"), lit("Venustiano Carranza"))
    .when(col("Donde Vivo").contains("chimalhuacán"), lit("Chimalhuacán"))
    .when(col("Donde Vivo").contains("ocoyoacac"), lit("Ocoyoacac"))
    .when(col("Donde Vivo").contains("tizayuca"), lit("Tizayuca"))
    .otherwise(None)
)

# Determine the state based on the extracted district, or fall back to state keywords
def build_expr_state(items, index=0, expr_state=None):
    if index >= len(items):
        return expr_state
    k, v = list(items)[index]
    if expr_state is None:
        expr_state = when(lower(col("current_district")) == k, v)
    else:
        expr_state = expr_state.when(lower(col("current_district")) == k, v)
    return build_expr_state(items, index + 1, expr_state)

expr_state = build_expr_state(list(district_to_state.items()))

current_residence_df_states = current_residence_df_disctricts.withColumn("current_state", expr_state)

# Drop the original and cleaned 'Donde Vivo' columns
current_residence_df = current_residence_df_states.drop("Donde Vivo")

current_residence_df.show()

## Trabajo

In [ ]:
# Convert the 'Trabajo' column to boolean and rename it
working_df_norm = current_residence_df.withColumn("is_working", when(lower(col("Trabajo")) == "si", True).otherwise(False))

# Drop the original 'Trabajo' column
working_df = working_df_norm.drop("Trabajo")

working_df.show()

## Dónde Trabajo

In [ ]:
workplace_df_div1 = working_df.withColumn("Dónde Trabajo", regexp_replace("Dónde Trabajo", "Trabajo en |una consultora llamada|", ""))
workplace_df_div2 = workplace_df_div1.withColumn("Dónde Trabajo", regexp_replace("Dónde Trabajo", ",.*|\\(.*| en.*|\"", ""))
workplace_df_trimmed = workplace_df_div2.withColumn("Dónde Trabajo", trim(col("Dónde Trabajo"))) # Add trim function
workplace_df = workplace_df_trimmed.withColumnRenamed("Dónde Trabajo", "workplace")
workplace_df.show()

## Semestre

In [ ]:
# Convert the column to lowercase for easier pattern matching
semester_df_lowcase = workplace_df.withColumn("Semestre_lower", lower(col("Semestre")))

# Extract the first numerical value or specific text patterns
semester_df_norm1 = semester_df_lowcase.withColumn("semester",
                  when(col("Semestre_lower").contains("terminado"), 11)
                  .when(col("Semestre_lower").contains("onceavo"), 11)
                  .when(col("Semestre_lower").contains("noveno"), 9)
                  .when(col("Semestre_lower").contains("decimo"), 10)
                  .when(col("Semestre_lower").contains("treceavo"), 13)
                  .otherwise(regexp_extract(col("Semestre_lower"), r"(\d+)", 1))) # Added r prefix here

# Cast to integer and replace values > 10 with 11
semester_df_norm2 = semester_df_norm1.withColumn("semester", col("semester").cast("int"))
semester_df_norm3 = semester_df_norm2.withColumn("semester", when(col("semester") > 10, 11).otherwise(col("semester")))

# Drop the original 'Semestre' column and the lowercase helper column
semester_df = semester_df_norm3.drop("Semestre", "Semestre_lower")

semester_df.show()

## Ingeniería

In [ ]:
# Rename the 'Ingeniería' column to 'engineering_definition'
ingineering_df = semester_df.withColumnRenamed("Ingeniería", "engineering_definition")

ingineering_df.show()

## Dato

In [ ]:
# Rename the 'Dato' column to 'data_definition'
data_df = ingineering_df.withColumnRenamed("Dato", "data_definition")

data_df.show()

## Artísta Favorito

In [ ]:
# Rename the 'Artista Favorito' column to 'favorite_artist'
artist_df = data_df.withColumnRenamed("Artista Favorito", "favorite_artist")

artist_df.show()

## Hobbies

In [ ]:
# Replace 'y' with ',' and then split the string by comma
hobbies_df_norm1 = artist_df.withColumn("3 hobbies_cleaned", regexp_replace(col("3 hobbies"), " y ", ","))
hobbies_df_norm2 = hobbies_df_norm1.withColumn("3 hobbies_cleaned", regexp_replace(col("3 hobbies_cleaned"), ", y ", ","))

# Split the cleaned string by comma and trim whitespace
split_hobbies = split(col("3 hobbies_cleaned"), ",")

# Create new columns for each hobbie, handling cases with less than 3 hobbies and capitalizing only the first letter
hobbies_df_split1 = hobbies_df_norm2.withColumn("hobbie_1", concat(initcap(substring(trim(split_hobbies.getItem(0)), 1, 1)), substring(trim(split_hobbies.getItem(0)), 2, 1000)))
hobbies_df_split2 = hobbies_df_split1.withColumn("hobbie_2", concat(initcap(substring(trim(split_hobbies.getItem(1)), 1, 1)), substring(trim(split_hobbies.getItem(1)), 2, 1000)))
hobbies_df_split3 = hobbies_df_split2.withColumn("hobbie_3", concat(initcap(substring(trim(split_hobbies.getItem(2)), 1, 1)), substring(trim(split_hobbies.getItem(2)), 2, 1000)))

# Drop the original '3 hobbies' column and the temporary cleaned column
hobbies_df = hobbies_df_split3.drop("3 hobbies", "3 hobbies_cleaned")

hobbies_df.show()

In [ ]:
hobbies_df.select("name", "hobbie_1", "hobbie_2", "hobbie_3").head(10)

## Con Cuantos Vivo

In [ ]:
# Convert the column to lowercase for easier pattern matching
residents_df_lowcase = hobbies_df.withColumn("Con cuantos vivo_lower", lower(col("Con cuantos vivo")))

# Extract the first numerical value
residents_df_ext1 = residents_df_lowcase.withColumn("number_of_residents_extracted", regexp_extract(col("Con cuantos vivo_lower"), r"(\d+)", 1))

# Cast the extracted string to integer, handling potential errors
residents_df_ext2 = residents_df_ext1.withColumn(
    "number_of_residents_int",
    when(col("number_of_residents_extracted") == "", None)
    .otherwise(col("number_of_residents_extracted").cast("int"))
)

# Apply the N+1 rule with exceptions and handle "nadie" case
residents_df_norm1 = residents_df_ext2.withColumn("number_of_residents",
                  when(col("Con cuantos vivo_lower").contains("nadie"), 1)
                  .when(col("Con cuantos vivo_lower").contains("contandome a mi") | col("Con cuantos vivo_lower").contains("somos"), col("number_of_residents_int"))
                  .when(col("number_of_residents_int").isNotNull(), col("number_of_residents_int") + 1)
                  .otherwise(None))

# Replacing the unique NULL value in the new 'number_of_residents' column with 3
residents_df_insert = residents_df_norm1.withColumn("number_of_residents", coalesce(col("number_of_residents"), lit(3)))

# Ensure the minimum value is 1
residents_df_norm2 = residents_df_insert.withColumn("number_of_residents", when(col("number_of_residents") < 1, 1).otherwise(col("number_of_residents")))

# Drop the original and temporary columns
residents_df = residents_df_norm2.drop("Con cuantos vivo", "Con cuantos vivo_lower", "number_of_residents_extracted", "number_of_residents_int")

residents_df.show()

## 3 Materias Favoritas

In [ ]:
residents_df.select("name", "3 Materias Favoritas").show()
residents_df.select("name", "3 Materias Favoritas").head(10)

In [ ]:
# Define the normalization dictionary
course_normalization = {
    'admin. de servicios de internet': 'Administración de Servicios de Internet',
    'administración de proyectos': 'Administración de Proyectos de Software',
    'algebra': 'Álgebra',
    'bases de datos': 'Bases de Datos',
    'bases de datos (todas)': 'Bases de Datos',
    'bases de datos distribuidas': 'Bases de Datos Distribuidas',
    'bd': 'Bases de Datos',
    'calculo y geometria analitica': 'Cálculo y Geometría Analítica',
    'calculo y geometría analítica': 'Cálculo y Geometría Analítica',
    'cálculo y geo.': 'Cálculo y Geometría Analítica',
    'cálculo y geometría analítica': 'Cálculo y Geometría Analítica',
    'cálculo y geometriía analítica': 'Cálculo y Geometría Analítica',
    'cálculo vectorial': 'Cálculo Vectorial',
    'cisco': 'Redes de Datos Seguras',
    'diseño digial moderno': 'Diseño Digital Moderno',
    'dispositivos': 'Dispositivos Electrónicos',
    'eda2': 'Estructura de Datos y Algoritmos II',
    'estadística': 'Fundamentos de Estadística',
    'estructura de datos y algoritmos': 'Estructura de Datos y Algoritmos I',
    'lenguajes autómatas': 'Lenguajes y Autómatas',
    'minería de datos': 'Minería de Datos',
    'poo': 'Programación Orientada a Objetos',
    'redes de datos seguras': 'Redes de Datos Seguras',
    'sistemas de comunicaciones': 'Sistemas de Comunicaciones',
    'sistemas distribuidos': 'Sistemas Distribuidos',
    'sistemas operativos': 'Sistemas Operativos',
    'taller socio-humanístico: liderazgo': 'Taller Sociohumanístico - Liderazgo',
}

# Create a UDF to split and normalize courses using recursion
def split_and_normalize_courses(courses_str):

    def normalize_course(course_clean, norm_dict_items, index=0):
        """Recursively search for normalization match"""
        if index >= len(norm_dict_items):
            return None

        key, value = norm_dict_items[index]
        if key == course_clean or key in course_clean or course_clean in key:
            return value

        return normalize_course(course_clean, norm_dict_items, index + 1)

    def normalize_courses(courses_list, index=0, result=None):
        """Recursively normalize a list of courses"""
        if result is None:
            result = []

        if index >= len(courses_list):
            return result

        course_clean = courses_list[index].strip()
        normalized = course_normalization.get(course_clean)

        if normalized is None:
            normalized = normalize_course(course_clean, list(course_normalization.items()))

        result.append(normalized)
        return normalize_courses(courses_list, index + 1, result)

    def split_by_y(course, parts=None, current_part=""):
        """Recursively split courses by 'y' handling edge cases"""
        if parts is None:
            parts = []

        if not course:
            if current_part:
                parts.append(current_part.strip())
            return parts

        if course.startswith(' y '):
            if current_part:
                parts.append(current_part.strip())
            return split_by_y(course[3:], parts, "")
        else:
            return split_by_y(course[1:], parts, current_part + course[0])

    def process_comma_split(courses_list, index=0, result=None):
        """Recursively process courses that were split by comma"""
        if result is None:
            result = []

        if index >= len(courses_list):
            return result

        course = courses_list[index]
        parts = split_by_y(course)

        if len(parts) > 2:
            # Rejoin first parts (course name) and keep last part separate
            result.append(' y '.join(parts[:-1]).strip())
            result.append(parts[-1].strip())
        else:
            result.extend([p.strip() for p in parts if p.strip()])

        return process_comma_split(courses_list, index + 1, result)

    def pad_list(lst, target_length, fill_value=None):
        """Recursively pad a list to target length"""
        if len(lst) >= target_length:
            return lst[:target_length]

        return pad_list(lst + [fill_value], target_length, fill_value)

    if courses_str is None:
        return [None, None, None]

    # Convert to lowercase for processing
    courses_lower = courses_str.lower().strip()

    # Replace periods with commas for uniform splitting
    courses_lower = courses_lower.replace('.', ',')

    # Split by comma first
    courses_list = [c.strip() for c in courses_lower.split(',') if c.strip()]

    # If we have fewer than 3 courses after splitting by comma, try splitting by 'y'
    if len(courses_list) < 3:
        courses_list = process_comma_split(courses_list)

    # Take only first 3 courses
    courses_list = courses_list[:3]

    # Normalize each course recursively
    normalized_courses = normalize_courses(courses_list)

    # Ensure we always return exactly 3 elements using recursive padding
    return pad_list(normalized_courses, 3)

# Register the UDF
split_normalize_udf = udf(split_and_normalize_courses, ArrayType(StringType()))

# Apply the UDF to create a temporary array column
courses_df_norm = residents_df.withColumn("courses_array", split_normalize_udf(col("3 Materias Favoritas")))

# Extract individual courses from the array
courses_df_split1 = courses_df_norm.withColumn("favorite_course_1", col("courses_array").getItem(0))
courses_df_split2 = courses_df_split1.withColumn("favorite_course_2", col("courses_array").getItem(1))
courses_df_split3 = courses_df_split2.withColumn("favorite_course_3", col("courses_array").getItem(2))

# Drop the temporary array column and original column
courses_df = courses_df_split3.drop("courses_array", "3 Materias Favoritas")

# Show the results
courses_df.select("name", "favorite_course_1", "favorite_course_2", "favorite_course_3").show(truncate=False)

In [ ]:
courses_df.select("name", "favorite_course_1", "favorite_course_2", "favorite_course_3").head(10)

# Resultado Final

In [ ]:
# Select and order the final columns
final_df = courses_df.select(
    "name",
    "age",
    "state",
    "date_of_birth",
    "grade_average",
    "current_district",
    "current_state",
    "is_working",
    "workplace",
    "semester",
    "engineering_definition",
    "data_definition",
    "favorite_artist",
    "hobbie_1",
    "hobbie_2",
    "hobbie_3",
    "number_of_residents",
    "favorite_course_1",
    "favorite_course_2",
    "favorite_course_3"
)

# Show the final DataFrame
final_df.show(truncate=False)

In [ ]:
# Export the final DataFrame to a Parquet file
final_df.write.parquet("cleaned_data.parquet", mode="overwrite")

# Read the Parquet file back into a DataFrame to display its content
parquet_df = spark.read.parquet("cleaned_data.parquet")

# Display the content of the DataFrame
parquet_df.show(truncate=False)